In [ ]:
# ============================================================
# Section 1: Imports and Test 2 settings
# ============================================================

from pathlib import Path
from datetime import datetime

import osxphotos

from explorephotoslibrary import *


USE_INVENTORY_CACHE = True

# For this run, rebuild backup because the backup Photos Library moved
# to a different external SSD path.
#
# After this run succeeds once, change this to:
# FORCE_REBUILD_INVENTORY_KEYS = set()
FORCE_REBUILD_INVENTORY_KEYS = set()

# Set to True only when you want to pick the Photos Library paths again.
# If False, Test 2 reuses paths saved in data/local_config/test2_library_paths.json.
FORCE_RESELECT_LIBRARY_PATHS = False


TEST2_LIBRARY_PROMPTS = {
    "backup_20250317": "Select BACKUP Photos Library: backup_20250317",
    "current_default": "Select CURRENT default Photos Library: current_default",
}


TEST2_DEFAULT_INITIAL_DIRS = {
    "backup_20250317": "/Volumes",
    "current_default": str(Path.home() / "Pictures"),
}

In [2]:
# ============================================================
# Section 2: Load or build inventories
# ============================================================

TEST2_LIBRARY_HISTORY_PATH = Path("data/local_config/test2_library_paths.json")


def get_test2_library_path(library_key):
    library_history = load_json_file(TEST2_LIBRARY_HISTORY_PATH, default={}) or {}

    saved_library_path = library_history.get(library_key)

    if (
        saved_library_path
        and Path(saved_library_path).exists()
        and not FORCE_RESELECT_LIBRARY_PATHS
    ):
        library_path = Path(saved_library_path)

        print("=" * 80)
        print(f"Use saved Photos Library path for: {library_key}")
        print("=" * 80)
        print(f"{library_key} library path:", library_path)
        print()

        return library_path

    if saved_library_path:
        initial_dir = Path(saved_library_path).parent
    else:
        initial_dir = Path(TEST2_DEFAULT_INITIAL_DIRS.get(library_key, "/Volumes"))

    prompt = TEST2_LIBRARY_PROMPTS.get(
        library_key,
        f"Select Photos Library for: {library_key}",
    )

    print("=" * 80)
    print(prompt)
    print("=" * 80)

    library_path = Path(
        choose_photos_library_path(
            initial_dir=initial_dir,
            prompt=prompt,
        )
    )

    library_history[library_key] = str(library_path)
    library_history[f"{library_key}_selected_at"] = datetime.now().isoformat()
    save_json_file(TEST2_LIBRARY_HISTORY_PATH, library_history)

    print(f"{library_key} library path:", library_path)
    print()

    return library_path


def load_or_build_inventory(library_key):
    library_path = get_test2_library_path(library_key)

    should_rebuild_inventory = library_key in FORCE_REBUILD_INVENTORY_KEYS

    if USE_INVENTORY_CACHE and not should_rebuild_inventory:
        print("=" * 80)
        print(f"Load inventory cache: {library_key}")
        print("=" * 80)

        try:
            inventory = load_inventory_cache(library_key)
            return inventory
        except FileNotFoundError:
            print(f"Cache not found for {library_key}. Build inventory instead.")
            print()

    if should_rebuild_inventory:
        print("=" * 80)
        print(f"Force rebuild inventory: {library_key}")
        print("=" * 80)
    else:
        print("=" * 80)
        print(f"Build inventory: {library_key}")
        print("=" * 80)

    osx_assets = osxphotos.PhotosDB(str(library_path)).photos()
    print(f"{library_key} osx asset count:", len(osx_assets))

    inventory = build_inventory(osx_assets)

    print()
    print(f"{library_key} inventory summary")
    print("-" * 80)
    print_inventory_summary(inventory)

    save_inventory_cache(inventory, library_key)

    return inventory


inventory_backup = load_or_build_inventory("backup_20250317")

print()

inventory_current = load_or_build_inventory("current_default")

Use saved Photos Library path for: backup_20250317
backup_20250317 library path: /Volumes/PRO-G40-0605/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary

Load inventory cache: backup_20250317
Cache not found for backup_20250317. Build inventory instead.

Build inventory: backup_20250317
backup_20250317 osx asset count: 71573
processed assets: 10000
processed assets: 20000
processed assets: 30000
processed assets: 40000
processed assets: 50000
processed assets: 60000
processed assets: 70000

backup_20250317 inventory summary
--------------------------------------------------------------------------------
inventory assets: 71573
inventory albums: 5170
inventory folders: 35
special assets:
  SCOPES_SYNDICATION: 0
movies: 6224
hidden: 0
favorites: 701
descriptions: 727
keywords: 23747
saved inventory cache: data/inventory_cache/backup_2025031

In [3]:
# ============================================================
# Section 3: Photo Library asset unique ID preflight
# ============================================================

import time

t0 = time.perf_counter()

fill_photo_library_asset_unique_ids(inventory_backup)

t1 = time.perf_counter()

fill_photo_library_asset_unique_ids(inventory_current)

t2 = time.perf_counter()

print("fill backup unique IDs elapsed seconds:", round(t1 - t0, 3))
print("fill current unique IDs elapsed seconds:", round(t2 - t1, 3))
print("fill total elapsed seconds:", round(t2 - t0, 3))
print()

backup_unique_id_ok = audit_photo_library_asset_unique_ids(
    inventory_backup,
    "BACKUP Photo Library asset unique ID audit",
)

print()

current_unique_id_ok = audit_photo_library_asset_unique_ids(
    inventory_current,
    "CURRENT DEFAULT Photo Library asset unique ID audit",
)

print()
print("backup_unique_id_ok:", backup_unique_id_ok)
print("current_unique_id_ok:", current_unique_id_ok)
print("ready_for_cross_library_comparison:", backup_unique_id_ok and current_unique_id_ok)

fill backup unique IDs elapsed seconds: 77.688
fill current unique IDs elapsed seconds: 86.523
fill total elapsed seconds: 164.211

BACKUP Photo Library asset unique ID audit
--------------------------------------------------------------------------------
total asset count: 71573
generated unique ID count: 71573
assets without unique ID: 0
duplicate unique ID group count: 0
duplicate asset count: 0
is Photo Library asset unique ID scheme unique: True

CURRENT DEFAULT Photo Library asset unique ID audit
--------------------------------------------------------------------------------
total asset count: 94939
generated unique ID count: 94939
assets without unique ID: 0
duplicate unique ID group count: 0
duplicate asset count: 0
is Photo Library asset unique ID scheme unique: True

backup_unique_id_ok: True
current_unique_id_ok: True
ready_for_cross_library_comparison: True


In [4]:
# ============================================================
# Section 4: Inventory comparison helpers
# ============================================================

from enum import Enum


class ChangeType(str, Enum):
    # Asset existence
    ASSET_MISSING_FROM_CURRENT = "ASSET_MISSING_FROM_CURRENT"
    ASSET_NEW_IN_CURRENT = "ASSET_NEW_IN_CURRENT"

    # Asset metadata fields
    ASSET_FIELD_CHANGED_DESCRIPTION = "ASSET_FIELD_CHANGED__description"
    ASSET_FIELD_CHANGED_KEYWORDS = "ASSET_FIELD_CHANGED__keywords"
    ASSET_FIELD_CHANGED_FAVORITE = "ASSET_FIELD_CHANGED__favorite"
    ASSET_FIELD_CHANGED_HIDDEN = "ASSET_FIELD_CHANGED__hidden"
    ASSET_FIELD_CHANGED_DATE = "ASSET_FIELD_CHANGED__date"
    ASSET_FIELD_CHANGED_DATE_ADDED = "ASSET_FIELD_CHANGED__date_added"
    ASSET_FIELD_CHANGED_ORIGINAL_FILENAME = "ASSET_FIELD_CHANGED__original_filename"
    ASSET_FIELD_CHANGED_IS_MOVIE = "ASSET_FIELD_CHANGED__is_movie"

    # Asset relationship metadata
    ASSET_ALBUM_MEMBERSHIP_REMOVED = "ASSET_ALBUM_MEMBERSHIP_REMOVED"
    ASSET_ALBUM_MEMBERSHIP_ADDED = "ASSET_ALBUM_MEMBERSHIP_ADDED"

    ASSET_FOLDER_PATHS_REMOVED = "ASSET_FOLDER_PATHS_REMOVED"
    ASSET_FOLDER_PATHS_ADDED = "ASSET_FOLDER_PATHS_ADDED"
    ASSET_FOLDER_PATHS_CHANGED = "ASSET_FOLDER_PATHS_CHANGED"

    # Album existence and folder relationship
    ALBUM_MISSING_FROM_CURRENT = "ALBUM_MISSING_FROM_CURRENT"
    ALBUM_NEW_IN_CURRENT = "ALBUM_NEW_IN_CURRENT"

    ALBUM_FOLDER_PATHS_REMOVED = "ALBUM_FOLDER_PATHS_REMOVED"
    ALBUM_FOLDER_PATHS_ADDED = "ALBUM_FOLDER_PATHS_ADDED"
    ALBUM_FOLDER_PATHS_CHANGED = "ALBUM_FOLDER_PATHS_CHANGED"

    ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS = "ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS"

    # Folder path existence
    FOLDER_PATH_MISSING_FROM_CURRENT = "FOLDER_PATH_MISSING_FROM_CURRENT"
    FOLDER_PATH_NEW_IN_CURRENT = "FOLDER_PATH_NEW_IN_CURRENT"


CHANGE_TYPE_DESCRIPTIONS = {
    ChangeType.ASSET_MISSING_FROM_CURRENT:
        "Asset exists in backup but not in the current default Photos Library. This is high-priority because it may mean the photo or video disappeared from the live iCloud library.",

    ChangeType.ASSET_NEW_IN_CURRENT:
        "Asset exists in the current default Photos Library but not in backup. Usually normal because current_default is later than the 2025-03-17 backup.",

    ChangeType.ASSET_FIELD_CHANGED_DESCRIPTION:
        "Same derived asset ID exists in both libraries, but the caption/description changed or disappeared.",

    ChangeType.ASSET_FIELD_CHANGED_KEYWORDS:
        "Same derived asset ID exists in both libraries, but keyword metadata changed or disappeared.",

    ChangeType.ASSET_FIELD_CHANGED_FAVORITE:
        "Same derived asset ID exists in both libraries, but favorite status changed. This may be real user action or metadata loss.",

    ChangeType.ASSET_FIELD_CHANGED_HIDDEN:
        "Same derived asset ID exists in both libraries, but hidden status changed. This may be real user action or metadata difference.",

    ChangeType.ASSET_FIELD_CHANGED_DATE:
        "Same derived asset ID exists in both libraries, but asset date changed. This is unusual and should be inspected carefully.",

    ChangeType.ASSET_FIELD_CHANGED_DATE_ADDED:
        "Same derived asset ID exists in both libraries, but date_added changed. This may be less important because import/sync timing can differ.",

    ChangeType.ASSET_FIELD_CHANGED_ORIGINAL_FILENAME:
        "Same derived asset ID exists in both libraries, but original filename changed. This is unusual and should be inspected carefully.",

    ChangeType.ASSET_FIELD_CHANGED_IS_MOVIE:
        "Same derived asset ID exists in both libraries, but is_movie changed. This is highly unusual.",

    ChangeType.ASSET_ALBUM_MEMBERSHIP_REMOVED:
        "Asset still exists in current, but one or more album memberships from backup are missing.",

    ChangeType.ASSET_ALBUM_MEMBERSHIP_ADDED:
        "Asset has album memberships in current that did not exist in backup. Often normal because current is later.",

    ChangeType.ASSET_FOLDER_PATHS_REMOVED:
        "Asset still exists in current, but backup folder-path relationships are gone in the current default Photos Library.",

    ChangeType.ASSET_FOLDER_PATHS_ADDED:
        "Asset has folder-path relationships in current that did not exist in backup. Often normal or caused by later organization.",

    ChangeType.ASSET_FOLDER_PATHS_CHANGED:
        "Asset still exists in both libraries, but folder-path relationships changed.",

    ChangeType.ALBUM_MISSING_FROM_CURRENT:
        "Album title exists in backup but not in the current default Photos Library. This may require album reconstruction.",

    ChangeType.ALBUM_NEW_IN_CURRENT:
        "Album title exists in the current default Photos Library but not in backup. Usually normal because current is later.",

    ChangeType.ALBUM_FOLDER_PATHS_REMOVED:
        "Album still exists in current, but it is no longer inside the folder path recorded in backup.",

    ChangeType.ALBUM_FOLDER_PATHS_ADDED:
        "Album gained folder-path relationships in current that did not exist in backup.",

    ChangeType.ALBUM_FOLDER_PATHS_CHANGED:
        "Album still exists in both libraries, but its folder path changed.",

    ChangeType.ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS:
        "Album title is duplicated or ambiguous in at least one inventory, so title-based comparison is not reliable for this album.",

    ChangeType.FOLDER_PATH_MISSING_FROM_CURRENT:
        "Folder path exists in backup but not in the current default Photos Library. Albums or assets may still exist elsewhere.",

    ChangeType.FOLDER_PATH_NEW_IN_CURRENT:
        "Folder path exists in the current default Photos Library but not in backup. Usually normal if the folder was created later.",
}


FIELD_CHANGE_TYPES = {
    "description": ChangeType.ASSET_FIELD_CHANGED_DESCRIPTION,
    "keywords": ChangeType.ASSET_FIELD_CHANGED_KEYWORDS,
    "favorite": ChangeType.ASSET_FIELD_CHANGED_FAVORITE,
    "hidden": ChangeType.ASSET_FIELD_CHANGED_HIDDEN,
    "date": ChangeType.ASSET_FIELD_CHANGED_DATE,
    "date_added": ChangeType.ASSET_FIELD_CHANGED_DATE_ADDED,
    "original_filename": ChangeType.ASSET_FIELD_CHANGED_ORIGINAL_FILENAME,
    "is_movie": ChangeType.ASSET_FIELD_CHANGED_IS_MOVIE,
}


# Photos UUID differs across Photos Libraries, so comparison uses
# photo_library_asset_unique_id.

def make_asset_index(inventory):
    # Build photo_library_asset_unique_id -> Asset object.
    #
    # Photos UUID is local to one Photos Library database.
    # It cannot be used to match the same asset across different
    # Photos Library currents or backups.
    #
    # This function is intentionally strict:
    # comparison must not silently include assets with None matching keys.
    index = {}
    assets_without_unique_id = []

    for asset in inventory["assets"]:
        unique_id = asset.get("photo_library_asset_unique_id")

        if unique_id is None:
            assets_without_unique_id.append(asset)
            continue

        if unique_id in index:
            existing_asset = index[unique_id]

            raise RuntimeError(
                "Duplicate photo_library_asset_unique_id found. "
                "Do not run comparison until the unique ID scheme is strengthened.\n\n"
                f"unique_id={unique_id!r}\n\n"
                f"existing uuid={existing_asset.get('uuid')}\n"
                f"existing filename={existing_asset.get('original_filename') or existing_asset.get('filename')}\n"
                f"existing path={existing_asset.get('path')}\n\n"
                f"new uuid={asset.get('uuid')}\n"
                f"new filename={asset.get('original_filename') or asset.get('filename')}\n"
                f"new path={asset.get('path')}"
            )

        index[unique_id] = asset

    if assets_without_unique_id:
        sample_lines = []

        for asset in assets_without_unique_id[:20]:
            sample_lines.append(
                "  "
                f"uuid={asset.get('uuid')} | "
                f"filename={asset.get('original_filename') or asset.get('filename')} | "
                f"asset_scope={asset.get('asset_scope')} | "
                f"syndicated={asset.get('syndicated')} | "
                f"saved_to_library={asset.get('saved_to_library')} | "
                f"path={asset.get('path')}"
            )

        raise RuntimeError(
            "Comparison input contains assets without photo_library_asset_unique_id.\n"
            "This means the preflight state and comparison state are not cleanly aligned.\n"
            "Do not continue comparison yet.\n\n"
            f"assets_without_unique_id={len(assets_without_unique_id)}\n\n"
            "First problematic assets:\n"
            + "\n".join(sample_lines)
        )

    return index


def make_album_title_index(inventory):
    # Build album_title -> list[Album object].
    # Album title may not be globally unique, so keep a list.
    index = {}

    for album in inventory["albums"].values():
        title = album["title"] or ""

        if title not in index:
            index[title] = []

        index[title].append(album)

    return index


def make_folder_path_index(inventory):
    # Build folder_path -> list[Folder object].
    # Folder path may theoretically collide, so keep a list.
    index = {}

    for folder in inventory["folders"].values():
        path = folder["path"] or ""

        if path not in index:
            index[path] = []

        index[path].append(folder)

    return index


def asset_display_name(asset):
    # Prefer original filename for human reading.
    if asset is None:
        return None

    return asset["original_filename"] or asset["filename"] or asset["uuid"]


def album_folder_paths(album):
    # Return sorted folder paths for one Album object.
    return sorted(
        folder["path"]
        for folder in album["folders"].values()
    )


def asset_album_titles(asset):
    # Return sorted album titles for one Asset object.
    return sorted(
        album["title"] or ""
        for album in asset["albums"].values()
    )


def asset_folder_paths(asset):
    # Return sorted folder paths for one Asset object.
    return sorted(
        folder["path"] or ""
        for folder in asset["folders"].values()
    )


def sort_asset_ids(asset_ids):
    # photo_library_asset_unique_id is a tuple and may contain mixed values.
    #
    # Example:
    #   ("IMG_1234.JPG", "01-01 12:00:00.000000", 1234567, None)
    #   ("IMG_1234.JPG", "01-01 12:00:00.000000", 1234567, (3024, 4032))
    #
    # Python cannot directly sort these because it eventually tries to compare:
    #   None < (3024, 4032)
    #
    # The ID itself is still valid as a dict/set key. This helper only creates
    # a stable debug/display order.
    return sorted(asset_ids, key=repr)


def add_diff(
    diff_records,
    change_type,
    scope,
    backup_object,
    current_object,
    backup_value,
    current_value,
    changed_field=None,
    note=None,
):
    # Add one normalized comparison record.
    if isinstance(change_type, ChangeType):
        change_type_value = change_type.value
        change_type_description = CHANGE_TYPE_DESCRIPTIONS.get(change_type)
    else:
        raise TypeError(f"change_type must be ChangeType, got: {change_type}")

    record = {
        "change_type": change_type_value,
        "change_type_description": change_type_description,
        "scope": scope,

        "changed_field": changed_field,

        "backup_object": backup_object,
        "current_object": current_object,

        "backup_value": backup_value,
        "current_value": current_value,

        "note": note,
    }

    diff_records.append(record)


def compare_asset_existence(inventory_backup, inventory_current, diff_records):
    # Compare derived asset ID existence.
    backup_assets = make_asset_index(inventory_backup)
    current_assets = make_asset_index(inventory_current)

    backup_ids = set(backup_assets)
    current_ids = set(current_assets)

    for asset_id in sort_asset_ids(backup_ids - current_ids):
        backup_asset = backup_assets[asset_id]

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ASSET_MISSING_FROM_CURRENT,
            scope="asset",
            backup_object=backup_asset,
            current_object=None,
            backup_value=asset_display_name(backup_asset),
            current_value=None,
            note="Asset exists in backup but not in the current default Photos Library. This is a high-priority possible iCloud crash data-loss case.",
        )

    for asset_id in sort_asset_ids(current_ids - backup_ids):
        current_asset = current_assets[asset_id]

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ASSET_NEW_IN_CURRENT,
            scope="asset",
            backup_object=None,
            current_object=current_asset,
            backup_value=None,
            current_value=asset_display_name(current_asset),
            note="Asset exists in the current default Photos Library but not in backup. This is usually normal because the current is later.",
        )


def compare_asset_metadata(inventory_backup, inventory_current, diff_records):
    # Compare metadata for assets with the same derived asset ID.
    backup_assets = make_asset_index(inventory_backup)
    current_assets = make_asset_index(inventory_current)

    common_ids = sort_asset_ids(set(backup_assets) & set(current_assets))

    fields_to_compare = [
        "original_filename",
        "is_movie",
        "date",
        "date_added",
        "description",
        "keywords",
        "favorite",
        "hidden",
    ]

    for asset_id in common_ids:
        backup_asset = backup_assets[asset_id]
        current_asset = current_assets[asset_id]

        for field in fields_to_compare:
            backup_value = backup_asset.get(field)
            current_value = current_asset.get(field)

            if backup_value != current_value:
                add_diff(
                    diff_records=diff_records,
                    change_type=FIELD_CHANGE_TYPES[field],
                    scope="asset",
                    backup_object=backup_asset,
                    current_object=current_asset,
                    backup_value=backup_value,
                    current_value=current_value,
                    changed_field=field,
                    note=f"Same derived asset ID but asset field changed: {field}",
                )


def compare_asset_album_membership(inventory_backup, inventory_current, diff_records):
    # Compare album membership by derived asset ID and album title.
    backup_assets = make_asset_index(inventory_backup)
    current_assets = make_asset_index(inventory_current)

    common_ids = sort_asset_ids(set(backup_assets) & set(current_assets))

    for asset_id in common_ids:
        backup_asset = backup_assets[asset_id]
        current_asset = current_assets[asset_id]

        backup_titles = set(asset_album_titles(backup_asset))
        current_titles = set(asset_album_titles(current_asset))

        removed_titles = sorted(backup_titles - current_titles)
        added_titles = sorted(current_titles - backup_titles)

        if removed_titles:
            add_diff(
                diff_records=diff_records,
                change_type=ChangeType.ASSET_ALBUM_MEMBERSHIP_REMOVED,
                scope="asset_album_membership",
                backup_object=backup_asset,
                current_object=current_asset,
                backup_value=removed_titles,
                current_value=None,
                note="Asset still exists, but some backup album memberships are missing from the current default Photos Library.",
            )

        if added_titles:
            add_diff(
                diff_records=diff_records,
                change_type=ChangeType.ASSET_ALBUM_MEMBERSHIP_ADDED,
                scope="asset_album_membership",
                backup_object=backup_asset,
                current_object=current_asset,
                backup_value=None,
                current_value=added_titles,
                note="Asset has album memberships in current that did not exist in backup. Often normal for later current.",
            )


def compare_asset_folder_paths(inventory_backup, inventory_current, diff_records):
    # Compare folder paths attached to the same asset.
    # These are derived through asset -> album_info -> folder_list.
    backup_assets = make_asset_index(inventory_backup)
    current_assets = make_asset_index(inventory_current)

    common_ids = sort_asset_ids(set(backup_assets) & set(current_assets))

    for asset_id in common_ids:
        backup_asset = backup_assets[asset_id]
        current_asset = current_assets[asset_id]

        backup_paths = set(asset_folder_paths(backup_asset))
        current_paths = set(asset_folder_paths(current_asset))

        if backup_paths == current_paths:
            continue

        if backup_paths and not current_paths:
            change_type = ChangeType.ASSET_FOLDER_PATHS_REMOVED
            note = "Asset still exists, but its folder paths are gone in the current default Photos Library."
        elif not backup_paths and current_paths:
            change_type = ChangeType.ASSET_FOLDER_PATHS_ADDED
            note = "Asset has folder paths in the current default Photos Library but did not have them in backup."
        else:
            change_type = ChangeType.ASSET_FOLDER_PATHS_CHANGED
            note = "Asset still exists, but folder paths changed."

        add_diff(
            diff_records=diff_records,
            change_type=change_type,
            scope="asset_folder_paths",
            backup_object=backup_asset,
            current_object=current_asset,
            backup_value=sorted(backup_paths),
            current_value=sorted(current_paths),
            note=note,
        )


def compare_album_existence_and_folder_paths(inventory_backup, inventory_current, diff_records):
    # Compare albums by album title.
    # Title is the human-facing identity; UUID may not be stable across libraries.
    backup_albums_by_title = make_album_title_index(inventory_backup)
    current_albums_by_title = make_album_title_index(inventory_current)

    backup_titles = set(backup_albums_by_title)
    current_titles = set(current_albums_by_title)

    for title in sorted(backup_titles - current_titles):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ALBUM_MISSING_FROM_CURRENT,
            scope="album",
            backup_object=backup_albums_by_title[title],
            current_object=None,
            backup_value=title,
            current_value=None,
            note="Album title exists in backup but not in the current default Photos Library. This may require album reconstruction.",
        )

    for title in sorted(current_titles - backup_titles):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ALBUM_NEW_IN_CURRENT,
            scope="album",
            backup_object=None,
            current_object=current_albums_by_title[title],
            backup_value=None,
            current_value=title,
            note="Album title exists in the current default Photos Library but not in backup. Usually normal for later current.",
        )

    for title in sorted(backup_titles & current_titles):
        backup_albums = backup_albums_by_title[title]
        current_albums = current_albums_by_title[title]

        if len(backup_albums) != 1 or len(current_albums) != 1:
            add_diff(
                diff_records=diff_records,
                change_type=ChangeType.ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS,
                scope="album",
                backup_object=backup_albums,
                current_object=current_albums,
                backup_value=len(backup_albums),
                current_value=len(current_albums),
                note="Album title is not unique in at least one inventory. Folder comparison by title is ambiguous.",
            )
            continue

        backup_album = backup_albums[0]
        current_album = current_albums[0]

        backup_paths = set(album_folder_paths(backup_album))
        current_paths = set(album_folder_paths(current_album))

        if backup_paths == current_paths:
            continue

        if backup_paths and not current_paths:
            change_type = ChangeType.ALBUM_FOLDER_PATHS_REMOVED
            note = "Album still exists, but it is no longer inside any folder path in the current default Photos Library."
        elif not backup_paths and current_paths:
            change_type = ChangeType.ALBUM_FOLDER_PATHS_ADDED
            note = "Album gained folder paths in the current default Photos Library."
        else:
            change_type = ChangeType.ALBUM_FOLDER_PATHS_CHANGED
            note = "Album still exists, but its folder path changed."

        add_diff(
            diff_records=diff_records,
            change_type=change_type,
            scope="album_folder_paths",
            backup_object=backup_album,
            current_object=current_album,
            backup_value=sorted(backup_paths),
            current_value=sorted(current_paths),
            note=note,
        )


def compare_folder_paths(inventory_backup, inventory_current, diff_records):
    # Compare folder paths by human-readable path.
    backup_folders_by_path = make_folder_path_index(inventory_backup)
    current_folders_by_path = make_folder_path_index(inventory_current)

    backup_paths = set(backup_folders_by_path)
    current_paths = set(current_folders_by_path)

    for path in sorted(backup_paths - current_paths):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.FOLDER_PATH_MISSING_FROM_CURRENT,
            scope="folder",
            backup_object=backup_folders_by_path[path],
            current_object=None,
            backup_value=path,
            current_value=None,
            note="Folder path exists in backup but not in the current default Photos Library. Albums/assets may still exist elsewhere.",
        )

    for path in sorted(current_paths - backup_paths):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.FOLDER_PATH_NEW_IN_CURRENT,
            scope="folder",
            backup_object=None,
            current_object=current_folders_by_path[path],
            backup_value=None,
            current_value=path,
            note="Folder path exists in the current default Photos Library but not in backup.",
        )


def compare_inventories(inventory_backup, inventory_current):
    # Run all comparison passes and return normalized diff records.
    diff_records = []

    compare_asset_existence(inventory_backup, inventory_current, diff_records)
    compare_asset_metadata(inventory_backup, inventory_current, diff_records)
    compare_asset_album_membership(inventory_backup, inventory_current, diff_records)
    compare_asset_folder_paths(inventory_backup, inventory_current, diff_records)
    compare_album_existence_and_folder_paths(inventory_backup, inventory_current, diff_records)
    compare_folder_paths(inventory_backup, inventory_current, diff_records)

    return diff_records


def summarize_diff_records(diff_records):
    # Count diff records by change_type.
    summary = {}

    for record in diff_records:
        change_type = record["change_type"]

        if change_type not in summary:
            summary[change_type] = 0

        summary[change_type] += 1

    return dict(sorted(summary.items()))

In [5]:
# ============================================================
# Section 5: Run inventory comparison summary
# ============================================================

if not (backup_unique_id_ok and current_unique_id_ok):
    raise RuntimeError(
        "Photo Library asset unique ID preflight failed. "
        "Do not run cross-library comparison yet."
    )

diff_records = compare_inventories(
    inventory_backup=inventory_backup,
    inventory_current=inventory_current,
)

summarize_diff_records(diff_records)


{'ALBUM_FOLDER_PATHS_CHANGED': 10,
 'ALBUM_FOLDER_PATHS_REMOVED': 308,
 'ALBUM_MISSING_FROM_CURRENT': 49,
 'ALBUM_NEW_IN_CURRENT': 842,
 'ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS': 25,
 'ASSET_ALBUM_MEMBERSHIP_ADDED': 609,
 'ASSET_ALBUM_MEMBERSHIP_REMOVED': 12218,
 'ASSET_FIELD_CHANGED__date': 74,
 'ASSET_FIELD_CHANGED__date_added': 7,
 'ASSET_FIELD_CHANGED__description': 3,
 'ASSET_FIELD_CHANGED__favorite': 16,
 'ASSET_FIELD_CHANGED__keywords': 3944,
 'ASSET_FOLDER_PATHS_CHANGED': 500,
 'ASSET_FOLDER_PATHS_REMOVED': 18763,
 'ASSET_MISSING_FROM_CURRENT': 16,
 'ASSET_NEW_IN_CURRENT': 23382,
 'FOLDER_PATH_MISSING_FROM_CURRENT': 5,
 'FOLDER_PATH_NEW_IN_CURRENT': 3}

In [ ]:
# ============================================================
# Section 5A: Inspect ASSET_MISSING_FROM_CURRENT records
# ============================================================

from pathlib import Path
import json
import csv
import os
from collections import Counter


TARGET_CHANGE_TYPE = "ASSET_MISSING_FROM_CURRENT"

missing_from_current_records = [
    record
    for record in diff_records
    if record.get("change_type") == TARGET_CHANGE_TYPE
]

print("=" * 120)
print("Section 5A: Inspect ASSET_MISSING_FROM_CURRENT records")
print("=" * 120)
print("matched record count:", len(missing_from_current_records))


def _asset_album_titles_for_print(asset):
    if asset is None:
        return []

    albums = asset.get("albums") or {}

    return sorted(
        album.get("title") or ""
        for album in albums.values()
    )


def _asset_folder_paths_for_print(asset):
    if asset is None:
        return []

    folders = asset.get("folders") or {}

    return sorted(
        folder.get("path") or folder.get("title") or ""
        for folder in folders.values()
    )


def _asset_file_exists(asset):
    if asset is None:
        return None

    path = asset.get("path")

    if path is None:
        return None

    return os.path.exists(path)


def _asset_file_size_on_disk(asset):
    if asset is None:
        return None

    path = asset.get("path")

    if path is None:
        return None

    try:
        return os.path.getsize(path)
    except OSError:
        return None


def _asset_identity_summary(asset):
    if asset is None:
        return None

    return {
        "photo_library_asset_unique_id_repr": repr(asset.get("photo_library_asset_unique_id")),
        "uuid": asset.get("uuid"),
        "filename": asset.get("filename"),
        "original_filename": asset.get("original_filename"),
        "is_movie": asset.get("is_movie"),

        "date": asset.get("date"),
        "date_added": asset.get("date_added"),
        "date_modified": asset.get("date_modified"),

        "path": asset.get("path"),
        "path_exists": _asset_file_exists(asset),
        "file_size_bytes": asset.get("file_size_bytes"),
        "file_size_on_disk": _asset_file_size_on_disk(asset),

        "adjustment_signature": repr(asset.get("adjustment_signature")),
        "hasadjustments": asset.get("hasadjustments"),
        "path_edited": asset.get("path_edited"),
        "path_edited_live_photo": asset.get("path_edited_live_photo"),
        "edited_duration_seconds": asset.get("edited_duration_seconds"),

        "width": asset.get("width"),
        "height": asset.get("height"),
        "original_width": asset.get("original_width"),
        "original_height": asset.get("original_height"),

        "description": asset.get("description"),
        "keywords": list(asset.get("keywords") or []),
        "favorite": asset.get("favorite"),
        "hidden": asset.get("hidden"),

        "albums": _asset_album_titles_for_print(asset),
        "folders": _asset_folder_paths_for_print(asset),

        "asset_scope": asset.get("asset_scope"),
        "syndicated": asset.get("syndicated"),
        "saved_to_library": asset.get("saved_to_library"),
        "ismissing": asset.get("ismissing"),
        "original_filesize": asset.get("original_filesize"),
        "path_derivatives": list(asset.get("path_derivatives") or []),
    }


missing_from_current_assets = [
    record.get("backup_object")
    for record in missing_from_current_records
]

missing_from_current_summaries = [
    {
        "index": index,
        "change_type": record.get("change_type"),
        "note": record.get("note"),
        "backup_asset": _asset_identity_summary(record.get("backup_object")),
        "current_asset": _asset_identity_summary(record.get("current_object")),
    }
    for index, record in enumerate(missing_from_current_records, start=1)
]


print()
print("Quick counts")
print("-" * 120)
print("is_movie:", dict(Counter(asset.get("is_movie") for asset in missing_from_current_assets if asset is not None)))
print("favorite:", dict(Counter(asset.get("favorite") for asset in missing_from_current_assets if asset is not None)))
print("hidden:", dict(Counter(asset.get("hidden") for asset in missing_from_current_assets if asset is not None)))
print("path_exists:", dict(Counter(_asset_file_exists(asset) for asset in missing_from_current_assets if asset is not None)))
print("asset_scope:", dict(Counter(asset.get("asset_scope") for asset in missing_from_current_assets if asset is not None)))


for item in missing_from_current_summaries:
    asset = item["backup_asset"]

    print()
    print("=" * 120)
    print(f'{item["index"]:02d}. {asset.get("original_filename") or asset.get("filename")}')
    print("=" * 120)

    print("change_type:", item["change_type"])
    print("note:", item["note"])
    print()
    print("BACKUP asset")
    print("-" * 120)
    print("uuid:", asset.get("uuid"))
    print("photo_library_asset_unique_id:", asset.get("photo_library_asset_unique_id_repr"))
    print("original_filename:", asset.get("original_filename"))
    print("filename:", asset.get("filename"))
    print("is_movie:", asset.get("is_movie"))
    print()
    print("date:", asset.get("date"))
    print("date_added:", asset.get("date_added"))
    print("date_modified:", asset.get("date_modified"))
    print()
    print("path_exists:", asset.get("path_exists"))
    print("path:", asset.get("path"))
    print("file_size_bytes:", asset.get("file_size_bytes"))
    print("file_size_on_disk:", asset.get("file_size_on_disk"))
    print()
    print("width x height:", asset.get("width"), "x", asset.get("height"))
    print("original_width x original_height:", asset.get("original_width"), "x", asset.get("original_height"))
    print("hasadjustments:", asset.get("hasadjustments"))
    print("adjustment_signature:", asset.get("adjustment_signature"))
    print("path_edited:", asset.get("path_edited"))
    print("edited_duration_seconds:", asset.get("edited_duration_seconds"))
    print()
    print("description:", asset.get("description"))
    print("keywords:", asset.get("keywords"))
    print("favorite:", asset.get("favorite"))
    print("hidden:", asset.get("hidden"))
    print()
    print("albums:")
    for album_title in asset.get("albums") or []:
        print("  -", album_title)
    print("folders:")
    for folder_path in asset.get("folders") or []:
        print("  -", folder_path)


Section 5A: Inspect ASSET_MISSING_FROM_CURRENT records
matched record count: 16

Quick counts
------------------------------------------------------------------------------------------------------------------------
is_movie: {False: 16}
favorite: {False: 16}
hidden: {False: 16}
path_exists: {True: 16}
asset_scope: {'NORMAL_ORIGINALS': 16}

01. IMG_0078.PNG
change_type: ASSET_MISSING_FROM_CURRENT
note: Asset exists in backup but not in the current default Photos Library. This is a high-priority possible iCloud crash data-loss case.

BACKUP asset
------------------------------------------------------------------------------------------------------------------------
uuid: E5FDDA31-773F-4327-8480-C554EC99DCEB
photo_library_asset_unique_id: ('IMG_0078.PNG', '12-15 14:01:41.00', 6855252, (1284, 2100))
original_filename: IMG_0078.PNG
filename: E5FDDA31-773F-4327-8480-C554EC99DCEB.png
is_movie: False

date: 2021-12-15T14:01:41+08:00
date_added: 2021-12-15T14:01:41.717324+08:00
date_modified: 2

In [7]:
# ============================================================
# Section 6: Inspect diff records by change type
# ============================================================

target_change_type = ChangeType.ASSET_MISSING_FROM_CURRENT.value

matched_records = [
    record
    for record in diff_records
    if record["change_type"] == target_change_type
]

print("target change type:", target_change_type)
print("matched record count:", len(matched_records))
print()

for record in matched_records[:20]:
    backup_asset = record["backup_object"]
    current_asset = record["current_object"]

    print("change_type:", record["change_type"])
    print("description:", record["change_type_description"])
    print("changed_field:", record["changed_field"])
    print("backup_value:", record["backup_value"])
    print("current_value:", record["current_value"])

    if backup_asset is not None:
        print("backup uuid:", backup_asset["uuid"])
        print("backup original_filename:", backup_asset["original_filename"])
        print("backup date:", backup_asset["date"])

    if current_asset is not None:
        print("current uuid:", current_asset["uuid"])
        print("current original_filename:", current_asset["original_filename"])
        print("current date:", current_asset["date"])

    print("-" * 80)


target change type: ASSET_MISSING_FROM_CURRENT
matched record count: 16

change_type: ASSET_MISSING_FROM_CURRENT
description: Asset exists in backup but not in the current default Photos Library. This is high-priority because it may mean the photo or video disappeared from the live iCloud library.
changed_field: None
backup_value: IMG_0078.PNG
current_value: None
backup uuid: E5FDDA31-773F-4327-8480-C554EC99DCEB
backup original_filename: IMG_0078.PNG
backup date: 2021-12-15T14:01:41+08:00
--------------------------------------------------------------------------------
change_type: ASSET_MISSING_FROM_CURRENT
description: Asset exists in backup but not in the current default Photos Library. This is high-priority because it may mean the photo or video disappeared from the live iCloud library.
changed_field: None
backup_value: IMG_0080.PNG
current_value: None
backup uuid: 24B8DA31-7833-4811-A4BF-3922C92E065F
backup original_filename: IMG_0080.PNG
backup date: 2021-12-15T14:02:19+08:00
----

In [8]:
# ============================================================
# Appendix A: Duplicate diagnostic archive
# 
# TEMP: Diagnose potential duplicate groups by SHA256
#       with full manual-review metadata
# ============================================================

import hashlib
import os
import time
from datetime import datetime


def compute_sha256_for_asset(asset, chunk_size=1024 * 1024):
    cached_sha256 = asset.get("content_sha256")
    if cached_sha256:
        return cached_sha256

    path = asset.get("path")

    if path is None:
        return None

    if not os.path.exists(path):
        return None

    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    digest = sha256.hexdigest()
    asset["content_sha256"] = digest
    return digest


def build_potential_duplicate_groups_by_unique_id(inventory):
    unique_id_to_assets = {}

    for asset in inventory["assets"]:
        unique_id = asset.get("photo_library_asset_unique_id")

        if unique_id is None:
            continue

        if unique_id not in unique_id_to_assets:
            unique_id_to_assets[unique_id] = []

        unique_id_to_assets[unique_id].append(asset)

    return {
        unique_id: assets
        for unique_id, assets in unique_id_to_assets.items()
        if len(assets) > 1
    }


def normalize_string_list(values):
    result = []

    if values is None:
        return result

    if isinstance(values, str):
        return [values]

    if isinstance(values, dict):
        iterable = values.values()
    elif isinstance(values, (list, tuple, set)):
        iterable = values
    else:
        return [str(values)]

    for item in iterable:
        if item is None:
            continue

        if isinstance(item, str):
            value = item
        elif isinstance(item, dict):
            value = (
                item.get("title")
                or item.get("name")
                or item.get("path")
                or item.get("folder_path")
                or item.get("album_path")
            )
        else:
            value = str(item)

        if value:
            result.append(value)

    return sorted(set(result))


def get_asset_album_titles(asset):
    albums = asset.get("albums")
    return normalize_string_list(albums)


def get_asset_folder_paths(asset):
    folders = asset.get("folders")
    folder_paths = normalize_string_list(folders)

    # Some inventory formats may store folder paths under different keys.
    extra_candidates = [
        asset.get("folder_paths"),
        asset.get("folder_path"),
        asset.get("album_folder_paths"),
    ]

    for candidate in extra_candidates:
        folder_paths.extend(normalize_string_list(candidate))

    return sorted(set(folder_paths))


def get_asset_keywords(asset):
    keywords = asset.get("keywords")
    return normalize_string_list(keywords)


def get_asset_description(asset):
    return (
        asset.get("description")
        or asset.get("caption")
        or asset.get("title")
        or ""
    )


def parse_date_added_for_sort(asset):
    date_added = asset.get("date_added")

    if not date_added:
        return datetime.max

    if isinstance(date_added, datetime):
        return date_added

    text = str(date_added)

    try:
        return datetime.fromisoformat(text.replace("Z", "+00:00"))
    except Exception:
        return datetime.max


def asset_metadata_signature(asset):
    return {
        "albums": tuple(get_asset_album_titles(asset)),
        "folders": tuple(get_asset_folder_paths(asset)),
        "keywords": tuple(get_asset_keywords(asset)),
        "description": get_asset_description(asset),
        "favorite": asset.get("favorite"),
        "hidden": asset.get("hidden"),
        "hasadjustments": asset.get("hasadjustments"),
        "adjustment_signature": asset.get("adjustment_signature"),
    }


def metadata_score(asset):
    return (
        len(get_asset_album_titles(asset)) * 10
        + len(get_asset_folder_paths(asset)) * 10
        + len(get_asset_keywords(asset)) * 5
        + (1 if get_asset_description(asset) else 0)
        + (1 if asset.get("favorite") else 0)
        + (1 if asset.get("hidden") else 0)
    )


def choose_representative_asset(assets):
    # Prefer metadata-rich assets; tie-break by earliest Date Added.
    return sorted(
        assets,
        key=lambda asset: (
            -metadata_score(asset),
            parse_date_added_for_sort(asset),
            asset.get("uuid") or "",
        ),
    )[0]


def print_asset_manual_review_block(asset, indent="  "):
    print(f"{indent}UUID:", asset.get("uuid"))
    print(f"{indent}Original File Name:", asset.get("original_filename"))
    print(f"{indent}Filename:", asset.get("filename"))
    print(f"{indent}Date:", asset.get("date"))
    print(f"{indent}Date Added:", asset.get("date_added"))
    print(f"{indent}File Size:", asset.get("file_size_bytes"))
    print(f"{indent}Has Adjustments:", asset.get("hasadjustments"))
    print(f"{indent}Adjustment Signature:", asset.get("adjustment_signature"))
    print(f"{indent}Width x Height:", asset.get("width"), "x", asset.get("height"))
    print(f"{indent}Original Width x Height:", asset.get("original_width"), "x", asset.get("original_height"))
    print(f"{indent}Albums:", get_asset_album_titles(asset))
    print(f"{indent}Folder Paths:", get_asset_folder_paths(asset))
    print(f"{indent}Keywords:", get_asset_keywords(asset))
    print(f"{indent}Description:", get_asset_description(asset))
    print(f"{indent}Favorite:", asset.get("favorite"))
    print(f"{indent}Hidden:", asset.get("hidden"))
    print(f"{indent}Path:", asset.get("path"))


def print_cleanup_recommendation(assets):
    metadata_signatures = [asset_metadata_signature(asset) for asset in assets]
    metadata_all_same = all(
        signature == metadata_signatures[0]
        for signature in metadata_signatures
    )

    representative = choose_representative_asset(assets)

    if metadata_all_same:
        print("Recommendation:")
        print("  Metadata appears identical.")
        print("  Keep earliest / representative asset:")
        print("   ", representative.get("uuid"))
        print("  Delete other duplicate asset(s):")
        for asset in assets:
            if asset is not representative:
                print("   ", asset.get("uuid"))
    else:
        print("Recommendation:")
        print("  Metadata differs across duplicate assets.")
        print("  Do NOT blindly delete.")
        print("  Suggested representative, based on richer metadata + earliest Date Added:")
        print("   ", representative.get("uuid"))
        print("  Before deleting others, manually confirm whether album/folder/keyword membership should be preserved.")


def diagnose_potential_duplicate_groups_with_sha256_and_metadata(
    inventory,
    label,
    max_true_duplicate_groups_to_print=50,
    max_key_collision_groups_to_print=20,
):
    start_time = time.perf_counter()

    potential_groups = build_potential_duplicate_groups_by_unique_id(inventory)

    true_content_duplicate_groups = []
    key_collision_groups = []
    sha_error_assets = []

    checked_asset_count = 0

    for unique_id, assets in potential_groups.items():
        sha256_to_assets = {}

        for asset in assets:
            checked_asset_count += 1
            sha256 = compute_sha256_for_asset(asset)

            if sha256 is None:
                sha_error_assets.append(asset)
                continue

            if sha256 not in sha256_to_assets:
                sha256_to_assets[sha256] = []

            sha256_to_assets[sha256].append(asset)

        duplicate_sha_groups = {
            sha256: sha_assets
            for sha256, sha_assets in sha256_to_assets.items()
            if len(sha_assets) > 1
        }

        if duplicate_sha_groups:
            for sha256, sha_assets in duplicate_sha_groups.items():
                true_content_duplicate_groups.append(
                    {
                        "unique_id": unique_id,
                        "sha256": sha256,
                        "assets": sha_assets,
                    }
                )

        if len(sha256_to_assets) > 1:
            key_collision_groups.append(
                {
                    "unique_id": unique_id,
                    "sha256_to_assets": sha256_to_assets,
                }
            )

    elapsed = time.perf_counter() - start_time

    print(label)
    print("-" * 120)
    print("potential duplicate unique_id group count:", len(potential_groups))
    print("checked asset count:", checked_asset_count)
    print("sha error asset count:", len(sha_error_assets))
    print("true content duplicate group count:", len(true_content_duplicate_groups))
    print("key collision group count:", len(key_collision_groups))
    print("elapsed seconds:", round(elapsed, 3))

    print()
    print("TRUE CONTENT DUPLICATE GROUPS — MANUAL REVIEW")
    print("-" * 120)

    for index, group in enumerate(true_content_duplicate_groups, start=1):
        if index > max_true_duplicate_groups_to_print:
            print("... more true content duplicate groups not printed")
            break

        assets_sorted = sorted(
            group["assets"],
            key=lambda asset: (
                parse_date_added_for_sort(asset),
                asset.get("uuid") or "",
            ),
        )

        print("=" * 120)
        print(f"Group {index:02d}")
        print("=" * 120)
        print("unique_id:", group["unique_id"])
        print("sha256:", group["sha256"])
        print("asset count:", len(assets_sorted))

        first_asset = assets_sorted[0]
        print("Original File Name:", first_asset.get("original_filename"))
        print("Date:", first_asset.get("date"))
        print("File Size:", first_asset.get("file_size_bytes"))
        print("Adjustment Signature:", first_asset.get("adjustment_signature"))

        union_albums = sorted(
            set(
                album
                for asset in assets_sorted
                for album in get_asset_album_titles(asset)
            )
        )
        union_folders = sorted(
            set(
                folder
                for asset in assets_sorted
                for folder in get_asset_folder_paths(asset)
            )
        )
        union_keywords = sorted(
            set(
                keyword
                for asset in assets_sorted
                for keyword in get_asset_keywords(asset)
            )
        )

        print("Union Albums:", union_albums)
        print("Union Folder Paths:", union_folders)
        print("Union Keywords:", union_keywords)

        print()
        print_cleanup_recommendation(assets_sorted)
        print()

        for asset_index, asset in enumerate(assets_sorted, start=1):
            print("-" * 120)
            print(f"Asset {asset_index}")
            print_asset_manual_review_block(asset, indent="  ")

        print()

    print()
    print("KEY COLLISION GROUPS")
    print("-" * 120)

    for index, group in enumerate(key_collision_groups, start=1):
        if index > max_key_collision_groups_to_print:
            print("... more key collision groups not printed")
            break

        print("=" * 120)
        print(f"Key Collision Group {index:02d}")
        print("=" * 120)
        print("unique_id:", group["unique_id"])
        print("sha256 count:", len(group["sha256_to_assets"]))

        for sha256, assets in group["sha256_to_assets"].items():
            print("  sha256:", sha256)
            print("  asset count:", len(assets))

            for asset in assets:
                print("    uuid:", asset.get("uuid"))
                print("    original_filename:", asset.get("original_filename"))
                print("    filename:", asset.get("filename"))
                print("    date:", asset.get("date"))
                print("    date_added:", asset.get("date_added"))
                print("    file_size_bytes:", asset.get("file_size_bytes"))
                print("    albums:", get_asset_album_titles(asset))
                print("    folder_paths:", get_asset_folder_paths(asset))
                print("    keywords:", get_asset_keywords(asset))
                print("    path:", asset.get("path"))

        print()

    return {
        "potential_groups": potential_groups,
        "true_content_duplicate_groups": true_content_duplicate_groups,
        "key_collision_groups": key_collision_groups,
        "sha_error_assets": sha_error_assets,
    }


backup_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256_and_metadata(
    inventory_backup,
    "BACKUP potential duplicate diagnostic with metadata",
)

print()

current_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256_and_metadata(
    inventory_current,
    "CURRENT potential duplicate diagnostic with metadata",
)

BACKUP potential duplicate diagnostic with metadata
------------------------------------------------------------------------------------------------------------------------
potential duplicate unique_id group count: 0
checked asset count: 0
sha error asset count: 0
true content duplicate group count: 0
key collision group count: 0
elapsed seconds: 0.066

TRUE CONTENT DUPLICATE GROUPS — MANUAL REVIEW
------------------------------------------------------------------------------------------------------------------------

KEY COLLISION GROUPS
------------------------------------------------------------------------------------------------------------------------

CURRENT potential duplicate diagnostic with metadata
------------------------------------------------------------------------------------------------------------------------
potential duplicate unique_id group count: 0
checked asset count: 0
sha error asset count: 0
true content duplicate group count: 0
key collision group count: 

In [9]:
# ============================================================
# Appendix B: Debug assets without unique ID
# 
# DEBUG: Dump assets without photo_library_asset_unique_id
# ============================================================

import os
import time
from collections import Counter

def debug_dump_assets_without_photo_library_asset_unique_id(inventory, label, max_print=80):
    missing_assets = [
        asset
        for asset in inventory["assets"]
        if asset.get("photo_library_asset_unique_id") is None
    ]

    reason_counter = Counter()

    print(label)
    print("-" * 120)
    print("assets without photo_library_asset_unique_id:", len(missing_assets))
    print()

    for asset in missing_assets:
        path = asset.get("path")
        original_filename = asset.get("original_filename")
        filename = asset.get("filename")
        date = asset.get("date")
        file_size_bytes = asset.get("file_size_bytes")
        adjustment_signature = asset.get("adjustment_signature")

        if original_filename is None and filename is None:
            reason_counter["missing filename and original_filename"] += 1

        if date is None:
            reason_counter["missing date"] += 1

        if path is None:
            reason_counter["path is None"] += 1
        elif not os.path.exists(path):
            reason_counter["path does not exist"] += 1

        if file_size_bytes is None:
            reason_counter["file_size_bytes is None"] += 1

        if adjustment_signature is None:
            reason_counter["adjustment_signature is None"] += 1

    print("reason counter:")
    for reason, count in reason_counter.most_common():
        print(f"  {reason}: {count}")

    print()
    print("missing asset details:")
    print("-" * 120)

    for index, asset in enumerate(missing_assets[:max_print], start=1):
        path = asset.get("path")

        print(f"{index:02d}.")
        print("  uuid:", asset.get("uuid"))
        print("  original_filename:", asset.get("original_filename"))
        print("  filename:", asset.get("filename"))
        print("  date:", asset.get("date"))
        print("  date_added:", asset.get("date_added"))
        print("  path:", path)
        print("  path_exists:", None if path is None else os.path.exists(path))
        print("  file_size_bytes:", asset.get("file_size_bytes"))
        print("  adjustment_signature:", asset.get("adjustment_signature"))
        print("  is_movie:", asset.get("is_movie"))
        print("  hasadjustments:", asset.get("hasadjustments"))
        print("  path_edited:", asset.get("path_edited"))
        print("  asset_scope:", asset.get("asset_scope"))
        print("  albums:", list((asset.get("albums") or {}).values()))
        print("  folders:", list((asset.get("folders") or {}).values()))
        print("-" * 120)

debug_dump_assets_without_photo_library_asset_unique_id(
    inventory_current,
    "CURRENT DEFAULT assets without photo_library_asset_unique_id",
)

CURRENT DEFAULT assets without photo_library_asset_unique_id
------------------------------------------------------------------------------------------------------------------------
assets without photo_library_asset_unique_id: 0

reason counter:

missing asset details:
------------------------------------------------------------------------------------------------------------------------
